In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt

import manifold_dynamics.paths as pth
import visionlab_utils.storage as vst

In [ ]:
target = '08.MF1.F' # this exists --> amarvi/manifold-dynamics/eigenspectra/08.MF1.F.pkl

topk_local = vst.fetch(f"{pth.OTHERS}/topk_vals.pkl")
with open(topk_local, "rb") as f:
    topk_vals = pickle.load(f)
top_k = int(topk_vals[target]["k"])

print(f"{pth.SAVEDIR}/eigenspectra/{target}.pkl")
eigen_local = vst.fetch(f"{pth.SAVEDIR}/eigenspectra/{target}.pkl")
with open(eigen_local, "rb") as f:
    payload = pickle.load(f)


eigenspectra = payload['eigenspectra']
transformed = {
    label: np.log1p(L)
    for label, L in eigenspectra.items()
}


normed = {}
for label, eig in transformed.items():
    max_lambda = np.nanmax(eig, axis=1, keepdims=True)
    # eig_norm = eig / max_lambda
    eig_norm = eig / np.nansum(eig, axis=1, keepdims=True)
    normed[label] = eig_norm

n_pcs = payload['n_pcs']

In [ ]:
shared_vmax = float(max(np.nanmax(Z) for Z in normed.values()))

fig_heat, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True, constrained_layout=True)
for ax, label in zip(axes, normed):
    Z = transformed[label]
    im = ax.imshow(
        Z,
        aspect="auto",
        origin="lower",
        interpolation="nearest",
        vmin=0.0,
        # vmax=shared_vmax,
    )
    # find time index of max per PC (row-wise)
    t_max = np.nanargmax(Z, axis=1)
    pcs = np.arange(Z.shape[0])

    # overlay red dots
    ax.scatter(t_max, pcs, color='red', s=20, zorder=3)

    ax.set_title(label)
    ax.set_xlabel('time bin')
    
axes[0].set_ylabel("PC index")
axes[0].set_yticks(range(n_pcs))
axes[0].set_yticklabels([f"PC{i + 1}" for i in range(n_pcs)])
fig_heat.colorbar(
    im,
    ax=axes,
    fraction=0.025,
    pad=0.02,
    label="log(lambda)",
)

In [ ]:
payload['n_pcs']